# 8장 Unsloth로 무료 Colab에서 학습 돌리기 (QLoRA, Qwen2.5-7B-Instruct 4bit)

내 데이터로 LLM 파인튜닝하기 8장의 실습 노트북이에요. 웹 챕터: https://math-ai-matrix-pi.vercel.app/tracks/finetune/08-unsloth-colab.html

- **이 노트북은 작성자가 돌려 보지 않았어요.** 출력 칸이 비어 있는 이유예요. 직접 돌려 나온 숫자를 `checks/ch08.csv`에 계산값과 나란히 적어 주세요.
- 흐름: GPU 확인 → 설치 → 설정 → 4bit 모델 불러오기 → LoRA 붙이기 → 데이터 → SFTTrainer → 학습 → 잰 값 기록 → 답 써 보기 → 저장
- 설치 칸과 API 사용법은 Unsloth 공식 노트북(`unslothai/notebooks`의 `Qwen2.5_(7B)-Alpaca.ipynb`, `Qwen2.5_Coder_(14B)-Conversational.ipynb`)을 2026-10-11에 읽고 따랐어요. 버전이 바뀌면 공식 노트북의 설치 칸으로 바꿔 넣으세요.
- 데이터는 1장의 가상 쇼핑몰 안내문 8개로 만든 가상 문의·답변 600쌍이에요. 내 데이터로 바꿀 때는 `make_pairs()`만 고치면 돼요.
- 단위: 이 장은 1GB = 10^9바이트예요. PyTorch 숫자를 1024^3으로 나누면 GiB라서 조금 작게 보여요. 아래 칸은 둘 다 찍어요.

In [ ]:
# 1. 받은 GPU 확인. 무료 Colab이 어떤 GPU를 주는지는 Google FAQ에 적혀 있지 않아요.
!nvidia-smi

## 2. 설치

Unsloth 공식 노트북의 설치 칸을 그대로 옮겼어요(2026-10-11 확인). 공식 노트북은 `transformers==4.57.6`, `trl==0.22.2`로 버전을 고정해요. 아래 칸의 `SFTTrainer` 쓰는 법도 그 버전에 맞춘 공식 노트북의 모양이에요.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
# 3. 버전 기록. checks/ch08.csv의 메모 칸에 같이 적어 두면 나중에 숫자를 비교할 수 있어요.
from unsloth import FastLanguageModel   # unsloth를 먼저 불러와야 패치가 적용돼요
import torch, transformers, trl, peft, bitsandbytes, unsloth
print("unsloth", unsloth.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| compute capability:", torch.cuda.get_device_capability(0))

## 4. 설정

실습 저장소 `data/scenario.json`과 같은 값이에요(테스트가 대조해요). OOM이 나면 `MICRO_BATCH`를 줄이고 그만큼 `GRAD_ACCUM`을 늘려 곱 16을 지켜요.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # NF4 + 두 번 양자화로 미리 줄여 둔 파일, 약 5.55GB
MAX_SEQ_LENGTH = 1024
MICRO_BATCH = 2
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 16
EPOCHS = 3
LEARNING_RATE = 2e-4
SEED = 3407
OPTIM = "adamw_8bit"
PAIRS_TOTAL = 600
VAL_PAIRS = 60

# 계산 모형 값(results/ch08.json). 아래에서 잰 값과 나란히 찍어요.
CALC = {"weights_gb": 5.547, "lora_params": 40370176, "total_gb": 8.15, "steps_per_epoch": 34}

In [ ]:
# 5. bf16을 쓸 수 있는지. T4는 compute capability 7.5라 bf16 하드웨어 지원 목록에 없어요(4장).
# TRL의 SFTConfig는 fp16을 안 적으면 bf16을 True로 정하니, 둘 다 직접 적어요.
major, minor = torch.cuda.get_device_capability(0)
USE_BF16 = major >= 8
USE_FP16 = not USE_BF16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"capability {major}.{minor} -> bf16={USE_BF16}, fp16={USE_FP16}")

def mem(label):
    a, rsv = torch.cuda.max_memory_allocated(), torch.cuda.max_memory_reserved()
    print(f"[{label}] allocated {a/1e9:.3f} GB ({a/2**30:.3f} GiB) | reserved {rsv/1e9:.3f} GB ({rsv/2**30:.3f} GiB)")
    return a, rsv

## 6. 4bit 모델 불러오기

`load_in_4bit=True`는 얼린 기본 모델을 4비트로 둬요(6장). Unsloth는 `load_in_8bit`, `load_in_16bit`, `full_finetuning` 옵션도 같은 자리에 둬요. 이 파일은 임베딩과 출력층(`lm_head`)을 16비트로 남겨서, 76억 개 × 0.5바이트 = 3.81GB가 아니라 약 5.55GB예요.

In [ ]:
torch.cuda.reset_peak_memory_stats()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = True,
)
load_alloc, load_rsv = mem("모델을 불러온 뒤")
print("계산한 가중치 크기:", CALC["weights_gb"], "GB")

## 7. LoRA 붙이기

7개 선형층 모두에 랭크 16 LoRA를 붙여요. 계산으로는 학습할 숫자가 40,370,176개(전체의 0.53%)예요.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("학습할 숫자:", f"{trainable:,}", "| 계산값:", f"{CALC['lora_params']:,}", "| 같은가:", trainable == CALC["lora_params"])

## 8. 데이터: 가상 문의·답변 600쌍

1장의 가상 쇼핑몰 안내문 8개로 만든 쌍이에요. 답변은 인사, 안내, 맺음말이 늘 같은 "우리 말투"예요. 내 데이터로 바꿀 때는 이 칸만 고치면 돼요.

In [ ]:
import random
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

DOCS = {
    "환불": "상품을 받은 날로부터 7일 안에 환불을 신청할 수 있어요. 환불 금액은 카드 결제 취소로 3영업일 안에 돌려드려요.",
    "배송비": "주문 금액이 3만 원 미만이면 배송비는 3,000원이고, 3만 원 이상이면 무료 배송이에요.",
    "배송 기간": "평일 오후 2시 전에 주문하면 당일 출고하고, 보통 1~2일 안에 도착해요. 제주와 도서 지역은 2~3일 더 걸려요.",
    "교환": "사이즈 교환은 1회 무료예요. 단순 변심으로 교환하면 왕복 배송비 6,000원을 내요.",
    "적립금": "구매 금액의 1%를 적립금으로 드려요. 적립금은 받은 날로부터 1년이 지나면 사라져요.",
    "쿠폰": "신규 가입 쿠폰은 가입 후 30일 안에 쓸 수 있고, 다른 쿠폰과 함께 쓸 수 없어요.",
    "보증": "제품 보증 기간은 구매일로부터 1년이에요. 보증 기간 안에 생긴 제품 불량은 무상 수리해요.",
    "운영 시간": "고객센터는 평일 오전 10시부터 오후 6시까지 전화 상담을 받아요. 점심시간은 낮 12시부터 1시까지예요.",
}
ASK = ["{item} {topic} 관련해서 여쭤볼게요.", "{item} 샀는데 {topic}은 어떻게 되나요?", "{topic} 기준이 궁금해요. {item} 주문했어요.",
       "{item} 주문 건인데 {topic} 알려 주세요.", "혹시 {topic} 규정 좀 알 수 있을까요? 물건은 {item}이에요."]
ITEMS = ["운동화", "후드티", "텀블러", "백팩", "블루투스 스피커", "청바지", "머그컵", "우산", "모자", "양말 세트",
         "노트북 파우치", "가습기", "선글라스", "티셔츠", "슬리퍼"]
SYSTEM = "당신은 작은 쇼핑몰의 고객 문의 답변 초안을 쓰는 직원이에요. 정해진 인사와 맺음말을 지켜 짧게 답해요."

def make_pairs():
    pairs = []
    for topic, text in DOCS.items():
        for ask in ASK:
            for item in ITEMS:
                q = ask.format(item=item, topic=topic)
                a = f"안녕하세요, 고객님. 문의 주셔서 고마워요.\n{text}\n더 궁금한 점이 있으면 편하게 말씀해 주세요. 고객지원팀 드림"
                pairs.append([{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": q},
                              {"role": "assistant", "content": a}])
    return pairs

pairs = make_pairs()
assert len(pairs) == PAIRS_TOTAL
random.Random(SEED).shuffle(pairs)
val_pairs, train_pairs = pairs[:VAL_PAIRS], pairs[VAL_PAIRS:]

tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(c, tokenize = False, add_generation_prompt = False) for c in batch["conversations"]]}
train_ds = Dataset.from_dict({"conversations": train_pairs}).map(to_text, batched = True)
val_ds = Dataset.from_dict({"conversations": val_pairs}).map(to_text, batched = True)
print(len(train_ds), len(val_ds))
print(train_ds[0]["text"])

### 이 가상 데이터로 잰 숫자를 읽을 때 주의할 점

- **검증 손실은 품질 지표가 아니에요.** 같은 주제의 답변이 모두 똑같아서, 검증 60쌍의 답변이 학습 쪽에 글자 그대로 들어 있어요. 검증 손실이 아주 낮게 나와도 "돌아간다"는 확인일 뿐이에요. 1장이 경고한 누수와 같은 상황이라, 진짜 평가는 3장 방식으로 나눈 실제 데이터로 9장에서 해요.
- **계산값과 바로 비교하지 마세요.** 가상 쌍은 실습이 가정한 평균 393.6토큰보다 훨씬 짧아서, 잰 메모리와 시간이 계산값(8.15GB, 에폭당 15.7분)보다 한참 작게 나올 거예요. 9번 칸에서 센 길이 분포를 실습 저장소 `data/scenario.json`의 `length_histogram`에 넣고 `experiments.py`를 다시 돌린 뒤 비교해요.
- **`train_runtime`에는 에폭마다 도는 검증 시간도 들어가요.** 그래서 13번 칸이 거꾸로 구한 활용률은 실제보다 조금 낮게 나와요.

In [ ]:
# 9. 토큰 길이를 세서 계산 모형의 가정(평균 393.6토큰)과 비교하고, 스텝 수를 미리 계산해요.
import math
lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"]]
print("최소", min(lens), "| 평균", round(sum(lens)/len(lens), 1), "| 최대", max(lens), "| 잘리는 쌍", sum(l > MAX_SEQ_LENGTH for l in lens))
steps_per_epoch = math.ceil(math.ceil(len(train_ds) / MICRO_BATCH) / GRAD_ACCUM)
print("에폭당 스텝", steps_per_epoch, "| 계산값", CALC["steps_per_epoch"], "| 전체", steps_per_epoch * EPOCHS)

## 10. SFTTrainer

공식 Unsloth 노트북처럼 `tokenizer`, `dataset_text_field`, `max_seq_length`를 `SFTTrainer`에 넘겨요. 최신 TRL 문서에서는 같은 일을 `processing_class`, `SFTConfig(max_length=...)`로 적어요. 체크포인트는 에폭마다 저장하고 최근 2개를 남겨요(기본 `save_steps=500`이면 102스텝짜리 학습은 중간 저장이 없어요). Colab 연결이 끊길 수 있으니 `OUTPUT_DIR`를 Google Drive로 바꿔 두면 안전해요.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
OUTPUT_DIR = "outputs"   # 예: Drive를 연결했다면 "/content/drive/MyDrive/ch08_outputs"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = MICRO_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        warmup_steps = math.ceil(0.05 * steps_per_epoch * EPOCHS),
        learning_rate = LEARNING_RATE,
        fp16 = USE_FP16,
        bf16 = USE_BF16,
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = steps_per_epoch,
        save_strategy = "steps",
        save_steps = steps_per_epoch,
        save_total_limit = 2,
        optim = OPTIM,
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = OUTPUT_DIR,
        report_to = "none",
    ),
)

In [ ]:
# 11. 답변 부분에만 손실을 걸어요. Qwen2.5 대화 템플릿(ChatML)의 표시를 직접 적어요.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
# 확인: 손실을 거는 토큰(-100이 아닌 칸)만 되살려 읽으면 답변만 남아야 해요.
ex = trainer.train_dataset[0]
kept = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print(tokenizer.decode(kept))

In [ ]:
# 12. 학습. 시작 전 최대 메모리 기록을 지우고, 끝난 뒤 잰 값과 계산값을 나란히 찍어요.
import time
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer_stats = trainer.train()
wall = time.time() - t0
alloc, rsv = mem("학습이 끝난 뒤(최대)")
runtime = trainer_stats.metrics["train_runtime"]
print("train_runtime", round(runtime, 1), "초 | 벽시계", round(wall, 1), "초 | 스텝", trainer.state.global_step)
print("계산한 합계:", CALC["total_gb"], "GB (출력 칸 상한 포함, CUDA 몫 제외)")

In [ ]:
# 13. 잰 시간으로 활용률(MFU)을 거꾸로 구해 실습의 가정값(0.1~0.3)과 비교해요.
N_PARAMS = 7615616512
tokens_per_epoch = sum(min(l, MAX_SEQ_LENGTH) for l in lens)      # 채운 칸은 빼고 센 실제 토큰
flops = 6 * N_PARAMS * tokens_per_epoch * EPOCHS                   # 체크포인팅을 켠 어림(앞 2N + 다시 2N + 뒤 2N)
mfu = flops / (65e12 * runtime)
print("실제 토큰/에폭", tokens_per_epoch, "| 어림 활용률", round(mfu, 3), "(빈칸 계산과 검증 시간이 runtime에 들어가 실제보다 낮게 나와요)")

In [ ]:
# 14. checks/ch08.csv에 붙여 넣을 한 줄
import datetime
row = [datetime.date.today().isoformat(), torch.cuda.get_device_name(0), MODEL_NAME, "QLoRA",
       MAX_SEQ_LENGTH, MICRO_BATCH, GRAD_ACCUM, LORA_R, "unsloth", CALC["total_gb"],
       round(alloc/1e9, 3), round(rsv/1e9, 3), trainable, trainer.state.global_step, round(runtime, 1), round(mfu, 3),
       f"unsloth {unsloth.__version__} / trl {trl.__version__}"]
print(",".join(str(x) for x in row))

In [ ]:
# 15. 검증 문의 두 개로 답을 써 봐요. 점수 매기기는 9장에서 해요.
FastLanguageModel.for_inference(model)
for conv in val_pairs[:2]:
    msgs = conv[:2]   # system + user
    inputs = tokenizer.apply_chat_template(msgs, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")
    out = model.generate(input_ids = inputs, max_new_tokens = 128, use_cache = True, do_sample = False)
    print("문의:", msgs[1]["content"])
    print("모델:", tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens = True))
    print("정답:", conv[2]["content"]); print("-" * 40)

In [ ]:
# 16. 저장: LoRA 어댑터만(수십 MB 수준). 합치기·GGUF는 10장에서 다뤄요.
model.save_pretrained("ch08_lora")
tokenizer.save_pretrained("ch08_lora")
!du -sh ch08_lora
# if False: model.save_pretrained_merged("ch08_merged_16bit", tokenizer, save_method = "merged_16bit")
# if False: model.save_pretrained_gguf("ch08_gguf", tokenizer, quantization_method = "q4_k_m")